# 03. Consumer PostgreSQL + pgvector
Consumer czyta wiadomości z Kafka i zapisuje je do PostgreSQL, rozszerzając zapis o embedding tytułu, zgodnie z ideą stream-to-database z laboratoriów [file:8][file:12].

In [ ]:
from kafka import KafkaConsumer
import psycopg2
import json
from datetime import datetime
from sentence_transformers import SentenceTransformer

consumer = KafkaConsumer(
    "transactions",
    bootstrap_servers="kafka_streaming_lab:9092",
    auto_offset_reset="earliest",
    enable_auto_commit=True,
    group_id="transactions-pg-group",
    value_deserializer=lambda m: json.loads(m.decode("utf-8"))
)

conn = psycopg2.connect(
    dbname="mydb",
    user="myuser",
    password="myuserpass",
    host="postgres_bank_lab",
    port=5432
)
cursor = conn.cursor()

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

insert_sql = """
INSERT INTO transactions (
    sender, receiver, amount, transaction_timestamp,
    device_sender, device_receiver, title, title_embedding
) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""

required_fields = {
    "sender",
    "receiver",
    "amount",
    "timestamp",
    "device_sender",
    "device_receiver",
    "title"
}

inserted = 0
skipped = 0
errors = 0

print("Kafka-to-Postgres processor running...")

try:
    for msg in consumer:
        data = msg.value

        if not isinstance(data, dict):
            skipped += 1
            print(f"[SKIP {skipped}] Message is not a dict: {data}")
            continue

        missing_fields = required_fields - set(data.keys())
        if missing_fields:
            skipped += 1
            print(f"[SKIP {skipped}] Missing fields {missing_fields}: {data}")
            continue

        try:
            embedding = model.encode(data["title"]).tolist()
            tx_timestamp = datetime.fromisoformat(data["timestamp"].replace(" ", ""))

            cursor.execute(insert_sql, (
                data["sender"],
                data["receiver"],
                data["amount"],
                tx_timestamp,
                data["device_sender"],
                data["device_receiver"],
                data["title"],
                embedding
            ))
            conn.commit()

            inserted += 1
            print(f"[OK {inserted}] Inserted into PostgreSQL: {data}")

        except Exception as e:
            conn.rollback()
            errors += 1
            print(f"[ERROR {errors}] Failed to process message: {data}")
            print(f"Reason: {e}")

except KeyboardInterrupt:
    print("Consumer stopped manually.")

finally:
    print("---- SUMMARY ----")
    print(f"Inserted: {inserted}")
    print(f"Skipped: {skipped}")
    print(f"Errors: {errors}")

    cursor.close()
    conn.close()
    consumer.close()

    print("Connections closed.")